# 第 12 章 · 索引与散射：按掩码挑、按下标取、按下标写回

本章回答：attention 里的 mask 怎么"遮"？`torch.where` 为什么比 Python 的 `if` 好？`gather` 和 `scatter_add` 各在什么场景用（取目标类的概率、MoE 合并专家输出）？`topk` 返回什么？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/12-index-scatter.md`（讲解、动画、练习都在讲义里）。对应原文：[00-overview.md](../00-overview.md) §4.3；官方文档 `torch.gather` / `Tensor.scatter_add_` / `torch.where`（〔在线〕）；Transformer 讲义 [第 17 章 因果 mask](../../../transformer-deep-dive/notes/lessons/17-causal-mask.md)；本章输出在本机 torch 2.15.0 上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 mask | 填 −∞ 再 softmax | `masked_softmax()` | 第 2 行 [0.0759, 0.9241, 0] |
| 第二步 where | 逐元素三选一 | `leaky_where()` | [−0.2, 1, 3]；布尔索引只剩 2 个 |
| 第三步 gather | out[i] = logp[i][tgt[i]] | `gather_target_logp()` | [−0.3567, −0.5108] |
| 第四步 scatter_add | 按下标累加 | `scatter_add_demo()` | [4, 4, 2] |
| 第五步 topk | 值和下标 | `topk_demo()` | [0.9, 0.5]、[3, 1] |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `gather_target_logp`**（来自 `ch12_index_scatter.py`）：下面要用到的小工具。

In [ ]:
def gather_target_logp():
    logp = torch.log(torch.tensor([[0.7, 0.2, 0.1], [0.1, 0.3, 0.6]]))
    tgt = torch.tensor([0, 2])
    return logp.gather(1, tgt.unsqueeze(1)).squeeze(1)

**函数 `leaky_where`**（来自 `ch12_index_scatter.py`）：下面要用到的小工具。

In [ ]:
def leaky_where():
    x = torch.tensor([-2.0, 1.0, 3.0])
    return torch.where(x > 0, x, 0.1 * x), x[x > 0]

**函数 `masked_softmax`**（来自 `ch12_index_scatter.py`）：下面要用到的小工具。

In [ ]:
def masked_softmax():
    s = torch.tensor([[2.0, 1.0, 0.0], [0.5, 3.0, 1.0]])
    keep = torch.tensor([[True, False, False], [True, True, False]])
    return s.masked_fill(~keep, float("-inf")).softmax(-1)

**函数 `scatter_add_demo`**（来自 `ch12_index_scatter.py`）：下面要用到的小工具。

In [ ]:
def scatter_add_demo():
    out = torch.zeros(3)
    idx = torch.tensor([0, 2, 0, 1])
    src = torch.tensor([1.0, 2.0, 3.0, 4.0])
    return out.scatter_add(0, idx, src), torch.zeros(3).index_add(0, idx, src)

**函数 `topk_demo`**（来自 `ch12_index_scatter.py`）：下面要用到的小工具。

In [ ]:
def topk_demo():
    return torch.tensor([0.1, 0.5, 0.2, 0.9]).topk(2)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 mask + softmax：\n", masked_softmax())

In [ ]:
print("\n第二步 where / 布尔索引：", leaky_where())

In [ ]:
print("\n第三步 gather 目标类 log 概率：", gather_target_logp())

In [ ]:
print("\n第四步 scatter_add / index_add：", scatter_add_demo())

In [ ]:
print("\n第五步 topk：", topk_demo())

## 3. 代码里才看得到的

1. **gather 的 index 要和输出同形状**：所以先 `unsqueeze(1)` 变成 (2, 1)。
2. **`scatter_add` 返回新张量，`scatter_add_` 原地改**：带下划线的都是原地版本。

## 4. 自己改一改

- 把 `scatter_add_demo` 的 idx 改成全 0，看结果。
- `topk(2, largest=False)` 取最小的两个。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> mask 填负无穷，softmax 后正好是零；
> where 三选一形状不变，布尔索引长度看数据；
> gather 按号取，scatter_add 按号加回，topk 给值又给号。